In [78]:
import numpy as np
import pandas as pd
import os
import sklearn as skl
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn import metrics 

based on TSNE/PCA results, the 0 risk class seems to be the most separated, while the 1, 2, and 3 risk classes are more mixed. 
The 3 risk class is more separated, but still fairly mixed with class 1 and 2.  

Based on this, we decided to separate the target class into two different splits for logistic regression. The first option is using   
class 0 ("Low" risk) as logreg class 0 and all other classes (1-3, "moderate" to "severe" risk) as logreg class 1.  
The second option is splitting the classes in half, so that "low" and "moderate" (0-1) are logreg class 0 and "high" and "severe" (2-3)
as logreg class 1.   

In [79]:
def readData(datafile):
    # get the current directory: directory name for the abs path of the curr file
    dirpath = os.getcwd()
    abspath = dirpath + "\\" + datafile

    # read data into a pandas dataframe. data file is comma separated so use read_csv
    df = pd.read_csv(abspath)
     
    return df

In [80]:
def trainTestSplit(features, target, randomstate=None):
    # TEST is completely unseen data (20%), TRAIN is split into TRAIN (0.8 of 0.8 = 64%) and VAL (16%)
    # x is the feature data and y is the target
    xtrain, xtest, ytrain, ytest = train_test_split(features, target, test_size=0.2, shuffle=True, random_state=randomstate)
    #xtrain, xval, ytrain, yval = train_test_split(xtrain, ytrain, test_size=0.2)

    # only scale the features!! the target has to be kept as discrete vals and scaling would make continuous
    scaler = skl.preprocessing.StandardScaler()
    xtrain = scaler.fit_transform(xtrain)
    xtest = scaler.fit_transform(xtest)

    return xtrain, xtest, ytrain, ytest


def runLogReg(xtrain, xtest, ytrain, ytest):
    # create the model and train
    # C is inverse of regularization, so large C means small regularization
    logReg = LogisticRegression(C=10, max_iter=1000)
    logReg.fit(xtrain, ytrain)

    # predict new labels: predict() returns just the predicted label, predict_proba() returns the probabilities
    # for all classes so a matrix of size nxk where n is data points and k is num target classes
    ypred = logReg.predict(xtest)
    yprob = logReg.predict_proba(xtest)[:,1] # grabbing just column 1 for the probabilities of the positive class

    # metrics
    # need actual class probabilities (predict_proba) for the roc_curve and need the roc curve outputs (false pos
    # rate and true pos rate) to get the auc. just using "standard" f1 for binary prediction
    fpr, tpr, _ = metrics.roc_curve(ytest, yprob) # needs pos class prediction probabilities and true labels
    auc = metrics.auc(fpr, tpr)
    f1 = metrics.f1_score(ytest, ypred)
    accuracy = metrics.accuracy_score(ytest, ypred)

    # return all the metrics and the pos class probabilities for different uses of the function
    return auc, f1, accuracy, yprob

In [81]:
# read in data
df = readData("GamingandMentalHealth_final.csv")

# map to split 1: "low"/0 risk is still 0, all other values are now 1
# map before converting to numpy/splitting into features and target to use pandas map func
map1 = {0:0, 1:1, 2:1, 3:1}
data1 = df.copy()
data1.iloc[:,-1] = data1.iloc[:, -1].map(map1)

# set up for test train split
data1 = data1.to_numpy()
features = data1[:,0:-1]
target = data1[:,-1].astype(int)

# get test/train split (and scale) and run the logisitic regression model
xtrain_1, xtest_1, ytrain_1, ytest_1 = trainTestSplit(features, target) # splits and scales
auc_one, f1_one, accuracy_one, _ = runLogReg(xtrain_1, xtest_1, ytrain_1, ytest_1) # runs model

# print metrics 
print(f"Logistic Regression model with target split 1 ('Low'=0, 'Moderate' and above=1)")
print(f"AUC: {auc_one:.6f} F1: {f1_one:.6f} Accuracy: {accuracy_one:.6f}\n")


# map to split 2: low/moderate=0, high/severe=1
map2 = {0:0, 1:0, 2:1, 3:1}
data2 = df.copy()
data2.iloc[:,-1] = data2.iloc[:, -1].map(map2)

# set up for test train split
data2 = data2.to_numpy()
features = data2[:,0:-1]
target = data2[:,-1].astype(int)

# run the logisitic regression model and print metrics
xtrain_2, xtest_2, ytrain_2, ytest_2 = trainTestSplit(features, target) # splits and scales
auc_two, f1_two, accuracy_two, _ = runLogReg(xtrain_2, xtest_2, ytrain_2, ytest_2)
print(f"Logistic Regression model with target split 2 ('Low'/'Moderate'=0, 'High'/'Severe'=1)")
print(f"AUC: {auc_two:.6f} F1: {f1_two:.6f} Accuracy: {accuracy_two:.6f}")

Logistic Regression model with target split 1 ('Low'=0, 'Moderate' and above=1)
AUC: 1.000000 F1: 0.994764 Accuracy: 0.995000

Logistic Regression model with target split 2 ('Low'/'Moderate'=0, 'High'/'Severe'=1)
AUC: 0.998966 F1: 0.976378 Accuracy: 0.985000


Since the metrics for both binary splits were really high, I'm also going to try multiclass logistic regression using the one-versus-all method.  

One binary logistic regression model is used per possible target class label, so 4 models for this case ("Low", "Moderate", "High", "Severe" target
values). The target vector used for each model has a binary mask applied for that class, so that every instance of the current class is 1 and all 
other instances are 0. The positive class probabilities are saved for all models and the predicted multiclass label for a given instance is from 
whatever model had the highest positive probability on that instance. 

In [82]:
data_mc = df.copy()
data_mc = data_mc.to_numpy()
features = data_mc[:,0:-1]
orig_target = data_mc[:,-1]

# get test train split: this needs to be the same across all the models so do it before applying binary masks
xtrain, xtest, ytrain, ytest = trainTestSplit(features, orig_target, randomstate=42)

# need four target arrays: one for each class where the current class is 1 and all other classes are 0
prob_array = []
for i in range(4):
    # for each class, create a new array which is just a binary mask for that class applied to the orig target
    # ex for class 2, all instances of class 2 are now 1 and the rest are now 0
    # have to do for both test and train since they're already splut
    ytest_class = np.where(ytest==i, 1, 0)
    ytrain_class = np.where(ytrain==i, 1, 0)

    # run logisitic regression model. don't need metrics for each individual model, just the pos class probabilities
    _, _, _, yprob_class = runLogReg(xtrain, xtest, ytrain_class, ytest_class)
    prob_array.append(yprob_class)

# the yprob_class array returned is JUST the positive class probabilities, so now stack into a matrix of
# all the POSITIVE class probabilities per class model (each row is a diff model, one for each class value)
prob_array = np.vstack(prob_array)

# find the actual labels for each sample by finding the max positive probability across classes 
# np.argmax with axis=0 returns a new array which is the column index of the highest value for that column (model/class with highest prob)
ypred = np.argmax(prob_array, axis=0)

# calculate metrics on the multiclass predictions: get both macro and weighted f1 score and accuracy
# not getting auc bc its more complicated with multiclass
f1_macro = metrics.f1_score(ytest, ypred, average='macro')
f1_weighted = metrics.f1_score(ytest, ypred, average='weighted')
acc = metrics.accuracy_score(ytest, ypred)

# print metrics!
print(f"Logisitic Regression model for Multiclass prediction (using all four classes for target)")
print(f"F1 macro: {f1_macro:.6f} F1 weighted: {f1_weighted:.6f} Accuracy: {acc}")

Logisitic Regression model for Multiclass prediction (using all four classes for target)
F1 macro: 0.775536 F1 weighted: 0.841668 Accuracy: 0.84
